Gold Layer

Step 1. gold_listings

In [0]:
from pyspark.sql.functions import col, size, split, regexp_replace
from pyspark.sql.functions import sum, count, avg

df_listings = spark.read.format("delta").load(
    "/Volumes/workspace/default/airbnb_silver_2/listings"
)

df_calendar = spark.read.format("delta").load(
    "/Volumes/workspace/default/airbnb_silver_2/calendar"
)

# 1. Aggregate calendar (availability metrics)
df_calendar_agg = df_calendar.groupBy("listing_id", "city").agg(
    (col("is_available")).alias("dummy")  # placeholder so Spark doesn't error
)

df_calendar_agg = df_calendar.groupBy("listing_id", "city").agg(
    (sum("is_available") / count("*")).alias("availability_rate"),
    avg("calendar_minimum_nights").alias("avg_min_nights"),
    avg("calendar_maximum_nights").alias("avg_max_nights")
)

# 2. Add amenities count
df_listings = df_listings.withColumn(
    "amenities_count",
    size(split(regexp_replace(col("amenities"), '[\\[\\]"]', ""), ","))
)

# 3. Join listings + calendar
df_gold_listings = df_listings.join(
    df_calendar_agg,
    on=["listing_id", "city"],
    how="left"
)

# 4. Select final columns
df_gold_listings = df_gold_listings.select(
    "listing_id",
    "city",
    "neighbourhood_cleansed",
    "property_type",
    "room_type",
    "accommodates",
    "bedrooms",
    "beds",
    "price",
    "minimum_nights",
    "availability_365",
    "availability_rate",
    "avg_min_nights",
    "avg_max_nights",
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_location",
    "review_scores_value",
    "host_is_superhost",
    "instant_bookable",
    "estimated_occupancy_l365d",
    "estimated_revenue_l365d",
    "amenities_count"
)

df_gold_listings.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/airbnb_gold_2/gold_listings")

In [0]:
display(df_gold_listings.groupBy("city").count())

city,count
chicago,7825
twin_cities,4761
nashville,6633
seattle,6221
austin,10517


In [0]:
display(df_gold_listings.limit(10))

listing_id,city,neighbourhood_cleansed,property_type,room_type,accommodates,bedrooms,beds,price,minimum_nights,availability_365,availability_rate,avg_min_nights,avg_max_nights,number_of_reviews,review_scores_rating,review_scores_location,review_scores_value,host_is_superhost,instant_bookable,estimated_occupancy_l365d,estimated_revenue_l365d,amenities_count
78884,austin,78704,Entire rental unit,entire home/apt,2,0.0,2.0,103.0,2,150,0.410958904109589,2.0547945205479454,90.0,219,4.79,4.97,4.78,t,t,84.0,8652.0,41
89475,austin,78757,Entire home,entire home/apt,8,4.0,6.0,167.0,45,319,0.873972602739726,42.32876712328767,1125.0,129,4.86,4.87,4.91,t,f,255.0,42585.0,58
277028,austin,78703,Entire home,entire home/apt,16,5.0,13.0,201.0,1,259,0.7095890410958904,1.284931506849315,1125.0,130,4.7,4.77,4.64,t,f,30.0,6030.0,64
329306,austin,78702,Private room in home,private room,2,1.0,1.0,68.0,2,234,0.6410958904109589,2.1397260273972605,1125.0,681,4.84,4.78,4.8,f,t,192.0,13056.0,35
355232,austin,78704,Entire home,entire home/apt,7,3.0,7.0,1000.0,1,365,1.0,1.0,7.0,5,5.0,4.8,4.8,f,f,6.0,6000.0,8
525231,austin,78757,Entire guesthouse,entire home/apt,5,2.0,2.0,127.0,2,153,0.4191780821917808,2.0,1125.0,152,4.91,4.84,4.86,t,f,120.0,15240.0,69
622495,austin,78705,Entire condo,entire home/apt,2,1.0,2.0,68.0,2,315,0.863013698630137,2.0,365.0,65,4.94,4.89,4.83,f,f,84.0,5712.0,46
644051,austin,78704,Entire home,entire home/apt,10,4.0,6.0,665.0,2,365,1.0,2.0,90.0,36,4.92,5.0,4.86,f,f,0.0,0.0,32
666768,austin,78704,Private room in cabin,private room,2,1.0,1.0,91.0,2,355,0.9726027397260274,2.0,1125.0,12,4.92,4.83,4.92,f,f,0.0,0.0,6
846780,austin,78704,Entire home,entire home/apt,5,2.0,2.0,146.0,3,259,0.7095890410958904,3.0,1125.0,230,4.96,4.94,4.89,f,f,0.0,0.0,50


Step 2. gold_reviews

In [0]:
df_reviews = spark.read.format("delta").load(
    "/Volumes/workspace/default/airbnb_silver_2/reviews"
)
df_listings = spark.read.format("delta").load(
    "/Volumes/workspace/default/airbnb_silver_2/listings"
)

df_gold_reviews = df_reviews.join(
    df_listings.select(
        "listing_id",
        "city",
        "neighbourhood_cleansed",
        "room_type",
        "price"
    ),
    on=["listing_id", "city"],
    how="inner"
)

df_gold_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/airbnb_gold_2/gold_reviews")

In [0]:
display(df_gold_reviews.limit(10))

listing_id,city,review_id,review_date,reviewer_id,reviewer_name,comments,source_file,ingestion_timestamp,neighbourhood_cleansed,room_type,price
6422,nashville,4435141,2013-05-05,5500449,Jamie,"We both said it.. Michele and her husband are top notch! They kind of spoiled it for all of the other airbnb hosts. :) Their house was beautiful and comfortable. The space was private. They were both very welcoming. Our stay in Nashville was incredible. Highly, Highly, Highly recommend! These two are amazing people!",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 6,private room,43.0
6422,nashville,149883342,2017-05-06,111640667,Jonas,We had a great stay at Michele's place. Due to delayed flights we arrived in the middle of the night but there were still no problem to get a hold of Michele's daughter Keezee so we could get in the house. Nice area close to Shelby Park which is a great place to take a walk and get away from the city life for a few minutes. This house is adorable! I would recommend this to anyone!,nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 6,private room,43.0
6422,nashville,190877177,2017-09-05,112733720,Karen,"Michele's home is quant, comfortable and clean. There is a beautiful park outside her back door that is wonderful for walking, running, biking or just setting and enjoying the beautiful scenery. Michele and her husband Collier were great hosts and made us feel comfortable from the moment we arrived. A great value in a great city.",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 6,private room,43.0
6422,nashville,213065301,2017-11-19,26496163,Ayako,"I had such a pleasant stay at Michele's place. I got there at night, but her house stood out in her neighborhood with a lot of lights. As soon as she opened the door, her warm smile welcomed me. I like her taste of decorating her house. Her house was conveniently located. I just took route #4 bus from downtown. Thank you very much, Michele! I highly recommend her place.",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 6,private room,43.0
39870,nashville,407605798,2019-02-02,239664477,Chris,Great location to the Green Hills area of Nashville. Very clean and easy to find with very good directions.,nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 25,private room,70.0
39870,nashville,443771188,2019-04-26,148233814,Lisa,"Excellent host, very informative, great location, would definitely stay here again & recommend to others if visiting Nashville",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 25,private room,70.0
39870,nashville,1255969445613343573,2024-09-28,237409326,Anna,"Great value, lovely neighborhood and host! Evelyn took us as a last minute reservation and was very kind and accommodating. Would definitely stay again!",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 25,private room,70.0
72906,nashville,53681427,2015-11-11,23429666,Charles,"Richard is a friendly host, and the apartment is clean, well-equipped and in a great location on a quiet street close to several fun neighborhoods (Belmont/Hllsborough Village and 12 South). It's also a short drive or ride to all the downtown activities.",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 18,entire home/apt,91.0
329997,nashville,404900622,2019-01-25,234244457,Viren,A clean feel at home place with all the amenities. And Rick added a personal touch to his service by providing an iron and an ironing board at the very last minute at my request. So thank you Rick. I visit Nashville every few months for work and I will stay again on my next trip at Rick's for sure.,nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 19,entire home/apt,103.0
329997,nashville,900083828078330146,2023-05-26,135892041,Eric,"This was a great space, we really just were here for the night. But did a quick walk downtown. Slept well and had everything we needed! Rick was super responsive and helpful!",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 19,entire home/apt,103.0


In [0]:
display(df_gold_reviews.groupBy("city").count())

city,count
seattle,534954
chicago,462336
twin_cities,278225
austin,587895
nashville,663232


Step 3. gold_calendar

In [0]:
from pyspark.sql.functions import col, size, split, regexp_replace
from pyspark.sql.functions import sum, count, avg

df_calendar = spark.read.format("delta").load(
    "/Volumes/workspace/default/airbnb_silver_2/calendar"
)

df_gold_calendar = df_calendar.groupBy("city").agg(
    avg("is_available").alias("avg_availability"),
    avg("calendar_minimum_nights").alias("avg_min_nights"),
    avg("calendar_maximum_nights").alias("avg_max_nights"),
    count("*").alias("total_records")
)

df_gold_calendar.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/airbnb_gold_2/gold_calendar")

In [0]:
display(df_gold_calendar)

city,avg_availability,avg_min_nights,avg_max_nights,total_records
chicago,0.630009535119442,15.028649001658763,306296.9824107881,3161995
twin_cities,0.6416854406665187,7.531306685845083,627.6437914944894,1941071
nashville,0.6853592541959024,6.79865964390245,4608759.9481802285,3446696
seattle,0.5703638869960918,11.723942448522443,658.4158066840542,2553540
austin,0.6440490908291666,8.30339517243514,8469798.056218848,3844547


In [0]:
df_gold_listings.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_listings")

df_gold_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_reviews")

df_gold_calendar.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_calendar")

Final Validation for Handoff

In [0]:
df_gold_listings = spark.read.format("delta").load("/Volumes/workspace/default/airbnb_gold_2/gold_listings")
df_gold_reviews = spark.read.format("delta").load("/Volumes/workspace/default/airbnb_gold_2/gold_reviews")
df_gold_calendar = spark.read.format("delta").load("/Volumes/workspace/default/airbnb_gold_2/gold_calendar")

In [0]:
display(df_gold_listings.groupBy("city").count())
display(df_gold_reviews.groupBy("city").count())
display(df_gold_calendar)

city,count
chicago,7825
twin_cities,4761
nashville,6633
seattle,6221
austin,10517


city,count
chicago,492377
twin_cities,300632
seattle,575718
austin,588211
nashville,784756


city,avg_availability,avg_min_nights,avg_max_nights,total_records
chicago,0.630009535119442,15.028649001658763,306296.9824107881,3161995
twin_cities,0.6416854406665187,7.531306685845083,627.6437914944894,1941071
nashville,0.6853592541959024,6.79865964390245,4608759.9481802285,3446696
seattle,0.5703638869960918,11.723942448522443,658.4158066840542,2553540
austin,0.6440490908291666,8.30339517243514,8469798.056218848,3844547


In [0]:
print(df_gold_listings.columns)
print(df_gold_reviews.columns)
print(df_gold_calendar.columns)

['listing_id', 'city', 'neighbourhood_cleansed', 'property_type', 'room_type', 'accommodates', 'bedrooms', 'beds', 'price', 'minimum_nights', 'availability_365', 'availability_rate', 'avg_min_nights', 'avg_max_nights', 'number_of_reviews', 'review_scores_rating', 'review_scores_location', 'review_scores_value', 'host_is_superhost', 'instant_bookable', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'amenities_count']
['listing_id', 'city', 'review_id', 'review_date', 'reviewer_id', 'reviewer_name', 'comments', 'source_file', 'ingestion_timestamp', 'neighbourhood_cleansed', 'room_type', 'price']
['city', 'avg_availability', 'avg_min_nights', 'avg_max_nights', 'total_records']


In [0]:
from pyspark.sql.functions import col, count, when

display(
    df_gold_listings.groupBy("city").agg(
        count("*").alias("total_rows"),
        count(when(col("price").isNull(), 1)).alias("null_price"),
        count(when(col("availability_rate").isNull(), 1)).alias("null_availability_rate"),
        count(when(col("amenities_count").isNull(), 1)).alias("null_amenities_count")
    )
)

city,total_rows,null_price,null_availability_rate,null_amenities_count
chicago,7825,0,0,0
twin_cities,4761,0,0,0
nashville,6633,0,0,0
seattle,6221,0,0,0
austin,10517,0,0,0


In [0]:
display(
    df_gold_reviews.groupBy("city").agg(
        count("*").alias("total_rows"),
        count(when(col("comments").isNull(), 1)).alias("null_comments"),
        count(when(col("price").isNull(), 1)).alias("null_price_after_join"),
        count(when(col("neighbourhood_cleansed").isNull(), 1)).alias("null_neighbourhood_after_join")
    )
)

city,total_rows,null_comments,null_price_after_join,null_neighbourhood_after_join
seattle,534954,0,0,0
chicago,462336,0,0,0
twin_cities,278225,0,0,0
austin,587895,0,0,0
nashville,663232,0,0,0


In [0]:
display(spark.table("workspace.default.gold_listings").limit(5))
display(spark.table("workspace.default.gold_reviews").limit(5))
display(spark.table("workspace.default.gold_calendar").limit(5))

listing_id,city,neighbourhood_cleansed,property_type,room_type,accommodates,bedrooms,beds,price,minimum_nights,availability_365,availability_rate,avg_min_nights,avg_max_nights,number_of_reviews,review_scores_rating,review_scores_location,review_scores_value,host_is_superhost,instant_bookable,estimated_occupancy_l365d,estimated_revenue_l365d,amenities_count
78884,austin,78704,Entire rental unit,entire home/apt,2,0.0,2.0,103.0,2,150,0.410958904109589,2.0547945205479454,90.0,219,4.79,4.97,4.78,t,t,84.0,8652.0,41
89475,austin,78757,Entire home,entire home/apt,8,4.0,6.0,167.0,45,319,0.873972602739726,42.32876712328767,1125.0,129,4.86,4.87,4.91,t,f,255.0,42585.0,58
277028,austin,78703,Entire home,entire home/apt,16,5.0,13.0,201.0,1,259,0.7095890410958904,1.284931506849315,1125.0,130,4.7,4.77,4.64,t,f,30.0,6030.0,64
329306,austin,78702,Private room in home,private room,2,1.0,1.0,68.0,2,234,0.6410958904109589,2.1397260273972605,1125.0,681,4.84,4.78,4.8,f,t,192.0,13056.0,35
355232,austin,78704,Entire home,entire home/apt,7,3.0,7.0,1000.0,1,365,1.0,1.0,7.0,5,5.0,4.8,4.8,f,f,6.0,6000.0,8


listing_id,city,review_id,review_date,reviewer_id,reviewer_name,comments,source_file,ingestion_timestamp,neighbourhood_cleansed,room_type,price
6422,nashville,4435141,2013-05-05,5500449,Jamie,"We both said it.. Michele and her husband are top notch! They kind of spoiled it for all of the other airbnb hosts. :) Their house was beautiful and comfortable. The space was private. They were both very welcoming. Our stay in Nashville was incredible. Highly, Highly, Highly recommend! These two are amazing people!",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 6,private room,43.0
6422,nashville,149883342,2017-05-06,111640667,Jonas,We had a great stay at Michele's place. Due to delayed flights we arrived in the middle of the night but there were still no problem to get a hold of Michele's daughter Keezee so we could get in the house. Nice area close to Shelby Park which is a great place to take a walk and get away from the city life for a few minutes. This house is adorable! I would recommend this to anyone!,nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 6,private room,43.0
6422,nashville,190877177,2017-09-05,112733720,Karen,"Michele's home is quant, comfortable and clean. There is a beautiful park outside her back door that is wonderful for walking, running, biking or just setting and enjoying the beautiful scenery. Michele and her husband Collier were great hosts and made us feel comfortable from the moment we arrived. A great value in a great city.",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 6,private room,43.0
6422,nashville,213065301,2017-11-19,26496163,Ayako,"I had such a pleasant stay at Michele's place. I got there at night, but her house stood out in her neighborhood with a lot of lights. As soon as she opened the door, her warm smile welcomed me. I like her taste of decorating her house. Her house was conveniently located. I just took route #4 bus from downtown. Thank you very much, Michele! I highly recommend her place.",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 6,private room,43.0
39870,nashville,407605798,2019-02-02,239664477,Chris,Great location to the Green Hills area of Nashville. Very clean and easy to find with very good directions.,nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 25,private room,70.0


city,avg_availability,avg_min_nights,avg_max_nights,total_records
chicago,0.630009535119442,15.028649001658763,306296.9824107881,3161995
twin_cities,0.6416854406665187,7.531306685845083,627.6437914944894,1941071
nashville,0.6853592541959024,6.79865964390245,4608759.9481802285,3446696
seattle,0.5703638869960918,11.723942448522443,658.4158066840542,2553540
austin,0.6440490908291666,8.30339517243514,8469798.056218848,3844547


In [0]:
df_listings = spark.read.format("delta").load(
    "/Volumes/workspace/default/airbnb_silver_2/listings"
)
df_reviews = spark.read.format("delta").load(
    "/Volumes/workspace/default/airbnb_silver_2/reviews"
)
display(
    df_reviews.select("listing_id", "city").distinct()
    .join(
        df_listings.select("listing_id", "city").distinct(),
        on=["listing_id", "city"],
        how="left_anti"
    )
)

listing_id,city
734118956626903937,nashville
4490882,nashville
727480927725293937,nashville
14631349,chicago
1242709,chicago
1136302059014250516,chicago
956032691663102717,nashville
1131314536973694948,nashville
48380346,nashville
1168306655988583984,nashville


In [0]:
from pyspark.sql.functions import col

df_calendar_silver = spark.read.format("delta").load(
    "/Volumes/workspace/default/airbnb_silver_2/calendar"
)

df_gold_calendar_daily = df_calendar_silver.select(
    col("listing_id"),
    col("city"),
    col("calendar_date"),
    col("is_available"),
    col("calendar_minimum_nights"),
    col("calendar_maximum_nights"),
    col("day_of_week"),
    col("is_weekend")
)

df_gold_calendar_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/airbnb_gold_2/gold_calendar_daily")

df_gold_calendar_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_calendar_daily")

In [0]:
display(df_gold_calendar_daily.groupBy("city").count())
display(df_gold_calendar_daily.limit(10))

city,count
chicago,3161995
twin_cities,1941071
nashville,3446696
seattle,2553540
austin,3844547


listing_id,city,calendar_date,is_available,calendar_minimum_nights,calendar_maximum_nights,day_of_week,is_weekend
5456,austin,2025-09-17,0,4,90,4,0
5456,austin,2025-09-18,0,4,90,5,0
5456,austin,2025-09-19,0,4,90,6,0
5456,austin,2025-09-20,0,4,90,7,1
5456,austin,2025-09-21,1,4,90,1,1
5456,austin,2025-09-22,1,4,90,2,0
5456,austin,2025-09-23,1,4,90,3,0
5456,austin,2025-09-24,1,4,90,4,0
5456,austin,2025-09-25,0,4,90,5,0
5456,austin,2025-09-26,0,4,90,6,0
